In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/chiragsharma1201/sentiment/sentiment_analysis.py
/kaggle/input/datasets/chiragsharma1201/code-compressor/cc_from_graph_and_ss.py
/kaggle/input/datasets/chiragsharma1201/steel-fault/nielit_group_project.py
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__results__.html
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/submission.csv
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__notebook__.ipynb
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__output__.json
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/model.png
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/custom.css
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__results___files/__results___24_0.png
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__results___files/__results___27_0.png
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__results___files/__results___30_0.png
/kaggle/input/notebooks/chi

In [2]:
!pip install -q transformers sentence-transformers faiss-cpu accelerate bitsandbytes
!pip install -q tree-sitter==0.20.4 tree-sitter-languages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.9/493.9 kB 8.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 90.9 MB/s eta 0:00:00:00:0100:01


In [3]:
import re
import os
import faiss
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from tree_sitter import Parser
from tree_sitter_languages import get_language
from kaggle_secrets import UserSecretsClient

#  HuggingFace token
secrets = UserSecretsClient()
hf_token = secrets.get_secret("hugging_face")

#  Load Mistral-7B locally on GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(" Loading Mistral-7B locally (~3-4 mins)...")

tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    token=hf_token
)
llm_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

tokenizer.pad_token = tokenizer.eos_token
llm_model.config.pad_token_id = llm_model.config.eos_token_id

#  Setup Tree-sitter
PY_LANGUAGE = get_language("python")
parser = Parser()
parser.set_language(PY_LANGUAGE)

print(" Mistral-7B loaded successfully!")

 Loading Mistral-7B locally (~3-4 mins)...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

 Mistral-7B loaded successfully!


In [4]:
print("Enter the path of your Python file:")
file_path = input(" File path: ").strip()


Enter the path of your Python file:


 File path:  /kaggle/input/datasets/chiragsharma1201/code-compressor/cc_from_graph_and_ss.py


In [5]:
def get_code_text(code, node):
    return code[node.start_byte:node.end_byte].decode("utf-8")


def extract_chunks_treesitter(src_code):
    src_bytes = src_code.encode()
    tree = parser.parse(src_bytes)
    root = tree.root_node

    chunks = {"functions": [], "classes": [], "imports": [], "calls": []}

    def traverse(node):
        t = node.type

        if t == "function_definition":
            name = node.child_by_field_name("name")
            chunks["functions"].append({
                "name": get_code_text(src_bytes, name),
                "type": "function",
                "start": node.start_point,
                "end": node.end_point,
                "code": get_code_text(src_bytes, node)
            })

        elif t == "class_definition":
            name = node.child_by_field_name("name")
            chunks["classes"].append({
                "name": get_code_text(src_bytes, name),
                "type": "class",
                "start": node.start_point,
                "end": node.end_point,
                "code": get_code_text(src_bytes, node)
            })

        elif t in ("import_statement", "import_from_statement"):
            chunks["imports"].append({
                "name": get_code_text(src_bytes, node).strip(),
                "type": "import",
                "start": node.start_point,
                "end": node.end_point,
                "code": get_code_text(src_bytes, node)
            })

        elif t == "call":
            fn = node.child_by_field_name("function")
            if fn:
                chunks["calls"].append({
                    "name": get_code_text(src_bytes, fn),
                    "type": "call",
                    "start": node.start_point,
                    "end": node.end_point,
                    "code": get_code_text(src_bytes, node)
                })

        for child in node.children:
            traverse(child)

    traverse(root)
    return chunks


def load_chunks_from_py(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        full_source = f.read()

    raw = extract_chunks_treesitter(full_source)

    # Only functions and classes for FAISS
    function_chunks = []
    for item in raw["functions"] + raw["classes"]:
        function_chunks.append(item)

    return function_chunks, raw


function_chunks, raw_chunks = load_chunks_from_py(file_path)

print(f" Extracted chunks:\n")
print(f"   Functions : {len(raw_chunks['functions'])}")
print(f"    Classes   : {len(raw_chunks['classes'])}")
print(f"   Imports   : {len(raw_chunks['imports'])}")
print(f"   Calls     : {len(raw_chunks['calls'])}")

 Extracted chunks:

   Functions : 20
    Classes   : 0
   Imports   : 17
   Calls     : 212


In [6]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

code_texts = [
    f"Type: {c['type']}\nName: {c['name']}\n\n{c['code']}"
    for c in function_chunks
]

embeddings = embedder.encode(code_texts, show_progress_bar=True)
print(f" Embeddings shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 Embeddings shape: (20, 384)


In [7]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print(f" Indexed {index.ntotal} chunks into FAISS")

 Indexed 20 chunks into FAISS


In [9]:
def call_mistral(system_prompt, user_prompt, max_tokens=600):
    """Central function to call Mistral locally"""
    prompt = f"""<s>[INST] {system_prompt}

{user_prompt} [/INST]"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    ).to("cuda")

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def ask_question(query, top_k=3, show_context=False):
    query_embedding = embedder.encode([query])
    D, I = index.search(np.array(query_embedding), top_k)
    retrieved_chunks = [function_chunks[idx] for idx in I[0]]

    context = "\n\n".join([
        f"[{c['type'].upper()}] {c['name']}:\n{c['code']}"
        for c in retrieved_chunks
    ])

    if show_context:
        print(" Retrieved Context:\n")
        print(context)
        print("\n" + "="*50 + "\n")

    system = """You are an expert Python code analyst.
Answer questions ONLY based on the provided code.
Be specific — reference exact function names, class names, and variable names.
If the answer is not in the code, say 'This is not covered in the provided code.'
Never hallucinate or invent information."""

    user = f"""Here is the relevant code from the project:

{context}

Question: {query}

Answer clearly and specifically, referencing the actual code above."""

    answer = call_mistral(system, user, max_tokens=600)
    return answer, retrieved_chunks


def generate_code(description, top_k=3):
    """Generate new code based on existing codebase style"""
    query_embedding = embedder.encode([description])
    D, I = index.search(np.array(query_embedding), top_k)
    retrieved_chunks = [function_chunks[idx] for idx in I[0]]

    context = "\n\n".join([
        f"[{c['type'].upper()}] {c['name']}:\n{c['code']}"
        for c in retrieved_chunks
    ])

    system = """You are an expert Python developer.
Generate clean, working Python code that matches the style and patterns of the existing codebase.
Always include docstrings and type hints.
Only generate what is asked — no extra explanation outside code comments."""

    user = f"""Here is the existing codebase for context and style reference:

{context}

Generate Python code for: {description}

Match the coding style, patterns, and conventions of the existing code above."""

    return call_mistral(system, user, max_tokens=800)


def debug_function(function_name):
    """Deep debug a specific function"""
    chunk = next((c for c in function_chunks if c["name"] == function_name), None)

    if not chunk:
        print(f" '{function_name}' not found.\n")
        print(" Available functions:")
        for c in function_chunks:
            print(f"  [{c['type'].upper()}] {c['name']}")
        return

    system = """You are a senior Python debugger and code reviewer.
Analyze code deeply and provide specific, actionable fixes.
Always provide the corrected version of the code."""

    user = f"""Debug and review this Python {chunk['type']}:

{chunk['code']}

Provide:
1.  Bugs & Logic Errors
2.  Edge Cases & Missing Checks
3.  Missing Error Handling
4.  Performance Issues
5.  Fixed & Improved Version of the code"""

    print(f"\n{'='*60}")
    print(f"🔍 Debug Report: `{function_name}`")
    print(f"{'='*60}\n")
    print(call_mistral(system, user, max_tokens=800))
    print(f"\n{'='*60}\n")


def explain_line(function_name, line_number):
    """Explain a specific line in a function"""
    chunk = next((c for c in function_chunks if c["name"] == function_name), None)

    if not chunk:
        print(f" '{function_name}' not found.")
        return

    lines = chunk['code'].split('\n')
    if line_number < 1 or line_number > len(lines):
        print(f" Line {line_number} doesn't exist. Function has {len(lines)} lines.")
        return

    target_line = lines[line_number - 1]

    system = "You are an expert Python teacher. Explain code clearly and simply."

    user = f"""In the function `{function_name}`:

Full function:
{chunk['code']}

Explain line {line_number} specifically:
{target_line}

Explain what this line does, why it's there, and how it fits in the overall function."""

    print(f"\n{'='*60}")
    print(f" Line {line_number} in `{function_name}`:")
    print(f"   {target_line.strip()}")
    print(f"{'='*60}\n")
    print(call_mistral(system, user, max_tokens=400))
    print(f"\n{'='*60}\n")


def summarize_file():
    snippets = "\n\n".join([
        f"[{c['type'].upper()}] {c['name']}:\n{c['code'][:300]}"
        for c in function_chunks
    ])

    system = "You are an expert Python engineer. Summarize code accurately based only on what you see. Do not guess."

    user = f"""Here are the functions and classes from a Python file:

{snippets}

Write a clear, accurate one paragraph summary of what this file does.
Only describe what you actually see in the code. Do not guess."""

    return call_mistral(system, user, max_tokens=200)


print(" File Summary:\n")
print(summarize_file())

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 File Summary:

This Python file contains various functions and a single class definition that work together to parse source code, construct a directed graph representing the relationships between different entities such as classes, functions, and variables within the code, and perform additional operations like finding the enclosing context of a given node, generating scoped names for variables, handling expressions, and traversing the parsed tree. The graph can then be visualized using tools like NetworkX and Graphviz, and optionally processed further through techniques like community detection algorithms or encoding using models like UniXCoder.


In [10]:
print(" RAG Code Assistant — Powered by Mistral-7B")
print("="*60)
print("Commands:")
print("  'exit'                    → Quit")
print("  'list'                    → Show all functions/classes")
print("  'summary'                 → Summarize the file")
print("  'debug:<function_name>'   → Deep debug a function")
print("  'explain:<name>:<line>'   → Explain a specific line")
print("  'generate:<description>'  → Generate new code")
print("  or just ask any question!")
print("="*60 + "\n")

while True:
    user_query = input(" You: ").strip()

    if not user_query:
        continue

    if user_query.lower() == "exit":
        print(" Goodbye!")
        break

    elif user_query.lower() == "list":
        print("\n Available functions/classes:")
        for c in function_chunks:
            print(f"  [{c['type'].upper()}] {c['name']}")

    elif user_query.lower() == "summary":
        print("\n File Summary:\n")
        print(summarize_file())

    elif user_query.lower().startswith("debug:"):
        func_name = user_query.split("debug:", 1)[1].strip()
        debug_function(func_name)

    elif user_query.lower().startswith("explain:"):
        # Format: explain:function_name:line_number
        parts = user_query.split(":")
        if len(parts) == 3:
            func_name = parts[1].strip()
            try:
                line_num = int(parts[2].strip())
                explain_line(func_name, line_num)
            except ValueError:
                print(" Format: explain:function_name:line_number")
        else:
            print(" Format: explain:function_name:line_number")

    elif user_query.lower().startswith("generate:"):
        description = user_query.split("generate:", 1)[1].strip()
        print("\n Generating code...\n")
        code = generate_code(description)
        print(f"\n{'='*60}")
        print(" Generated Code:")
        print(f"{'='*60}\n")
        print(code)
        print(f"\n{'='*60}\n")

    else:
        answer, retrieved = ask_question(user_query, top_k=3)
        print("\n Retrieved Chunks:")
        for r in retrieved:
            print(f"  [{r['type'].upper()}] {r['name']}")
        print(f"\n Answer:\n{answer}")

    print("\n" + "-"*60 + "\n")

 RAG Code Assistant — Powered by Mistral-7B
Commands:
  'exit'                    → Quit
  'list'                    → Show all functions/classes
  'summary'                 → Summarize the file
  'debug:<function_name>'   → Deep debug a function
  'explain:<name>:<line>'   → Explain a specific line
  'generate:<description>'  → Generate new code
  or just ask any question!



 You:  list



 Available functions/classes:
  [FUNCTION] get_code_text
  [FUNCTION] add_node
  [FUNCTION] add_edge
  [FUNCTION] extract_chunks_and_graph
  [FUNCTION] get_name
  [FUNCTION] enclosing_context
  [FUNCTION] get_scoped_var_name
  [FUNCTION] handle_variable_reference
  [FUNCTION] handle_expression
  [FUNCTION] traverse
  [FUNCTION] visualize_with_graphviz
  [FUNCTION] nx_to_igraph
  [FUNCTION] apply_leiden
  [FUNCTION] visualize_with_graphviz
  [FUNCTION] unixcoder_encoder
  [FUNCTION] apply_edge_similarity_unixcoder
  [FUNCTION] print_sorted_edge_scores
  [FUNCTION] retrieve_optimal_subgraph_k_and_x_hop
  [FUNCTION] build_class_method_map
  [FUNCTION] prune_code_from_subgraph

------------------------------------------------------------



 You:  summary


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

 You:  Provide me suggestions how to enhance this project if possible ?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



 Retrieved Chunks:
  [FUNCTION] visualize_with_graphviz
  [FUNCTION] visualize_with_graphviz
  [FUNCTION] prune_code_from_subgraph

 Answer:
Based on the provided code, here are some potential enhancement ideas:

1. Improve error handling: The `visualize_with_graphviz` function currently raises exceptions when there's an issue saving the image file. Instead, consider implementing more detailed error messages and providing users with alternative ways to handle these errors, such as logging or returning informative error codes.

2. Support different file formats: Currently, the `visualize_with_graphviz` function saves graphs as .png files. Consider adding support for other popular formats like SVG, JPG, PDF, etc., which can be achieved by changing the write_method argument passed to the write_x method.

3. Implement filtering options: In the `prune_code_from_subgraph` function, you could add optional arguments allowing users to specify which types of nodes they want to include or exclud

 You:  exit


 Goodbye!
